In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

In [ ]:
rs = 100 # Globalizando random state

In [ ]:
dadosNacional = pd.read_parquet("../outputs/dadosFinal.parquet")
dadosNorte = pd.read_parquet("../outputs/dadosNorte.parquet")
dadosNordeste = pd.read_parquet("../outputs/dadosNordeste.parquet")
dadosCentroOeste = pd.read_parquet("../outputs/dadosCentroOeste.parquet")
dadosSudeste = pd.read_parquet("../outputs/dadosSudeste.parquet")
dadosSul = pd.read_parquet("../outputs/dadosSul.parquet")

In [ ]:
bancos = {
    "Nacional": dadosNacional,
    "Norte": dadosNorte,
    "Nordeste": dadosNordeste,
    "Centro-Oeste": dadosCentroOeste,
    "Sudeste": dadosSudeste,
    "Sul": dadosSul
}

In [ ]:
dadosVarBin = {}

for regiao, dados in bancos.items():

    dados["atraso60Binario"] = dados["atraso60"].map({
        "Pontual": 0,
        "Atraso": 1
    })

    y = dados["atraso60Binario"]

    X = dados.drop(columns=[
        "atraso60",
        "atraso60Binario"
    ])

    dadosVarBin[regiao] = {
        "dados": dados,
        "X": X,
        "y": y
    }

In [ ]:
for regiao, dados in dadosVarBin.items():
    X = dados["X"]
    
    categoricas = X.select_dtypes(include=['object', 'category']).columns
    numericas = X.select_dtypes(exclude=['object', 'category']).columns
    
    dados["categoricas"] = categoricas
    dados["numericas"] = numericas

In [ ]:
pipeCategoricas = Pipeline([
    ('onehot', OneHotEncoder(handle_unknown = 'ignore')) # Transforma as colunas em números para a RF conseguir interpretar. handle unknown ignora colunas desconhecidas
])

preprocessador = ColumnTransformer([
    ('cat', pipeCategoricas, categoricas),
    ('num', 'passthrough', numericas) # Não faz nada com as numéricas
])

arvoreEstocastica = Pipeline([
    ('preprocessador', preprocessador),
    ('rf', RandomForestClassifier(
        n_jobs = -1,
        random_state = rs
    ))
])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = rs, stratify = y)

In [ ]:
arvoreEstocastica.fit(X_train, y_train) #Treinando o modelo
pred = arvoreEstocastica.predict_proba(X_test)[:,1]
print("Acurácia: ", roc_auc_score(y_test, pred))

In [ ]:
amostra = X_test.sample(10000, random_state=rs).index

resultadoPermutacao = permutation_importance(
    arvoreEstocastica,
    X_test.loc[amostra],
    y_test.loc[amostra],
    n_repeats = 3,
    random_state = rs,
    n_jobs = 1
)

In [ ]:
nomeVariavel = ( # Nomes das colunas depois do OneHotEncoder
    arvoreEstocastica.named_steps['preprocessador']
    .get_feature_names_out()
)

featureImportance = pd.DataFrame({ # Cria o DataFrame com os nomes e valores de importância associados
    "variavel": X_test.columns,
    "importancia": resultadoPermutacao.importances_mean
})

featureImportance["nome"] = ( # Limpa os nomes das variáveis
    featureImportance["variavel"]
    .str.replace("cat__", "", regex=False)
    .str.replace("num__", "", regex=False)
)

fiTopo = featureImportance.sort_values(by="importancia", ascending=False).head(20)

fiTopo = fiTopo.sort_values(by="importancia", ascending=True)

plt.figure(figsize=(10, 8))
plt.barh(fiTopo["nome"], fiTopo["importancia"], color='skyblue')
plt.xlabel("Aumento no Erro (Permutation Importance)")
plt.title("20 Variáveis mais relevantes para o atraso no tratamento")
plt.tight_layout()
plt.show()